# TP3 – Nettoyage et données Silver

Ryma Dinari – Retail Intelligence Platform

Étapes 4 à 6 de l'UC-06 : je définis mes règles de nettoyage à partir de ce que j'ai vu dans le notebook 01, je les applique, puis j'enregistre les données propres dans la zone Silver (en Parquet).

## 1. Mes règles de nettoyage

| Anomalie | Règle | Action | Justification |
|---|---|---|---|
| Tout est en texte dans les fichiers | Chaque colonne prend le type défini dans `config.py` | Conversion avec `try_cast` | Les types ont été vérifiés dans le notebook 01 (aucune valeur non convertible) |
| Espaces / champs vides | On enlève les espaces autour du texte, un champ vide devient null | `trim` | Éviter que "Books" et "Books " soient deux catégories différentes |
| Clé obligatoire vide ou invalide (date, produit, magasin, ticket, quantité, montant) | La ligne est mise à l'écart | Copie dans `data/quarantine`, pas supprimée | Sans ces infos on ne peut pas utiliser la vente ; on garde une trace |
| Clé facultative vide (promo, adresse, client démo.) | On garde la ligne | Valeur laissée à null | Ces colonnes ne servent pas à mes indicateurs, supprimer ferait perdre des ventes |
| Doublons ticket + produit | On garde | Aucune | Ce sont des vraies ventes (quantités et prix différents) |
| Lignes totalement identiques | Supprimées si présentes | `dropDuplicates` | Ce serait une vraie copie ; il y en a 0 aujourd'hui |
| Marge négative, vente à 0 € | On garde | Aucune | Montants cohérents, c'est le générateur de données ; ce n'est pas une erreur |
| Libellés manquants (marque, description) | On garde | Valeur laissée à null | On ne remplace pas une valeur sans savoir laquelle mettre |
| `s_closed_date_sk` incohérente (1917 à 2141) | Colonne retirée du Silver | Supprimée de `stores` | Dates impossibles, elle reste dans le fichier brut si besoin |
| `d_holiday` = Y partout | Colonne retirée du Silver | Supprimée de `dates` | Aucune information utile |

Je n'ai pas fait de règle pour les dates incohérentes dans les ventes : toutes les dates de vente existent dans le calendrier et sont entre 2001 et 2005.

En plus, j'ajoute deux colonnes à chaque table pour garder la provenance : le fichier d'origine et la date du traitement.

## 2. Préparation

In [1]:
import sys
sys.path.insert(0, "..")

from config import *
from pyspark.sql import functions as F

spark = creer_session()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 13:57:59 WARN Utils: Your hostname, rymadinari-VivoBook-ASUSLaptop-M1702QA-M1702QA, resolves to a loopback address: 127.0.1.1; using 10.188.47.136 instead (on interface wlp1s0)
26/11/18 13:57:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 13:58:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/18 13:58:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
def typer(table):
    """Lit le fichier brut, enlève les espaces et convertit chaque colonne dans son type."""
    df = lire_brut(spark, table)
    colonnes = []
    for c, type_cible in COLONNES[table]:
        texte = F.trim(F.col(c))
        texte = F.when(texte == "", None).otherwise(texte)      # champ vide -> null
        colonnes.append(texte.try_cast(type_cible).alias(c))
    return df.select(colonnes)


def ajouter_provenance(df, table):
    return (df.withColumn("source_fichier", F.lit(str(chemin_source(table))))
              .withColumn("date_traitement", F.current_timestamp()))


def separer(df, obligatoires):
    """Sépare les lignes valides et les lignes à mettre à l'écart (une clé obligatoire est nulle)."""
    condition_ok = F.lit(True)
    for c in obligatoires:
        condition_ok = condition_ok & F.col(c).isNotNull()
    return df.filter(condition_ok), df.filter(~condition_ok)

## 3. Ventes (store_sales)

In [3]:
ventes = typer("store_sales")
nb_avant = ventes.count()

obligatoires = ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number",
                "ss_quantity", "ss_sales_price", "ss_net_paid"]
ventes_ok, ventes_ecart = separer(ventes, obligatoires)

ventes_ok = ventes_ok.dropDuplicates()       # seulement les lignes 100 % identiques
ventes_ok = ajouter_provenance(ventes_ok, "store_sales")

nb_ok = ventes_ok.count()
nb_ecart = ventes_ecart.count()
print("lignes lues         :", nb_avant)
print("lignes gardées      :", nb_ok)
print("lignes mises à part :", nb_ecart)
print("doublons supprimés  :", nb_avant - nb_ok - nb_ecart)

lignes lues         : 667579
lignes gardées      : 667579
lignes mises à part : 0
doublons supprimés  : 0


In [4]:
ventes_ok.printSchema()

root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid: decimal(7,2) (nullable = 

Toutes les ventes passent les règles : aucune ligne mise à l'écart et aucun doublon complet. C'était attendu vu l'exploration. Les montants sont maintenant en décimal et les clés en entier.

## 4. Produits (item)

In [5]:
produits = typer("item")
produits_ok, produits_ecart = separer(produits, ["i_item_sk", "i_category"])
produits_ok = ajouter_provenance(produits_ok, "item")

print("produits lus     :", produits.count())
print("produits gardés  :", produits_ok.count())
print("mis à l'écart    :", produits_ecart.count())
produits_ok.select("i_item_sk", "i_category", "i_class", "i_current_price").show(5)

produits lus     : 17820
produits gardés  : 17820


mis à l'écart    : 0
+---------+-----------------+--------------------+---------------+
|i_item_sk|       i_category|             i_class|i_current_price|
+---------+-----------------+--------------------+---------------+
|        0|Sports & Outdoors|            Fan Shop|          72.29|
|        1|            Books|                 New|          95.67|
|        2|            Books|Education & Refer...|          62.74|
|        3|            Books|Education & Refer...|          98.67|
|        4|            Books|          NO_SUB_CAT|          40.95|
+---------+-----------------+--------------------+---------------+
only showing top 5 rows


## 5. Magasins (store)

In [6]:
magasins = typer("store").drop("s_closed_date_sk")
magasins_ok, magasins_ecart = separer(magasins, ["s_store_sk", "s_store_name"])
magasins_ok = ajouter_provenance(magasins_ok, "store")

print("magasins gardés :", magasins_ok.count(), "/ mis à l'écart :", magasins_ecart.count())
magasins_ok.select("s_store_sk", "s_store_name", "s_number_employees", "s_state").show()

magasins gardés : 12 / mis à l'écart : 0
+----------+------------+------------------+-------+
|s_store_sk|s_store_name|s_number_employees|s_state|
+----------+------------+------------------+-------+
|         0|      Forest|               217|     CO|
|         1|    Hillside|               244|     TN|
|         2|    Sherwood|               266|     CA|
|         3|     Pioneer|               221|     TX|
|         4|     Elmwood|               254|     KY|
|         5|       Egypt|               215|     KY|
|         6|  Bloomfield|               225|     IN|
|         7|     Buffalo|               225|     KS|
|         8|Williamstown|               298|     TX|
|         9|  Germantown|               213|     WV|
|        10|      Avalon|               254|     TN|
|        11|     Bedford|               221|     KY|
+----------+------------+------------------+-------+



## 6. Calendrier (date_dim)

In [7]:
dates = typer("date_dim").drop("d_holiday")
dates_ok, dates_ecart = separer(dates, ["d_date_sk", "d_date"])
dates_ok = ajouter_provenance(dates_ok, "date_dim")

print("dates gardées :", dates_ok.count(), "/ mises à l'écart :", dates_ecart.count())

dates gardées : 109573 / mises à l'écart : 0


Pour les produits, les magasins et les dates, rien n'a été mis à l'écart non plus. Les seuls changements sont les types, les espaces, les deux colonnes retirées et la provenance.

## 7. Écriture dans Silver

J'enregistre en Parquet dans `data/silver/<domaine>`. J'utilise des noms en anglais au pluriel (`sales`, `products`, `stores`, `dates`), comme dans l'exemple du sujet.
Le mode `overwrite` remplace l'ancien résultat : c'est OK ici parce que ce dossier ne contient que mes sorties, jamais les données sources.

In [8]:
silver = {
    "sales": ventes_ok,
    "products": produits_ok,
    "stores": magasins_ok,
    "dates": dates_ok,
}
for nom, df in silver.items():
    df.write.mode("overwrite").parquet(str(SILVER / nom))
    print("écrit :", SILVER / nom)

# lignes mises à l'écart (seulement s'il y en a)
ecarts = {"sales": ventes_ecart, "products": produits_ecart,
          "stores": magasins_ecart, "dates": dates_ecart}
for nom, df in ecarts.items():
    if df.count() > 0:
        df.write.mode("overwrite").parquet(str(QUARANTAINE / nom))
        print("quarantaine :", QUARANTAINE / nom)

écrit : /home/rymadinari/TP3/data/silver/sales


écrit : /home/rymadinari/TP3/data/silver/products


écrit : /home/rymadinari/TP3/data/silver/stores


écrit : /home/rymadinari/TP3/data/silver/dates


## 8. Contrôles après écriture

In [9]:
for nom, df in silver.items():
    relu = spark.read.parquet(str(SILVER / nom))
    memes_types = relu.dtypes == df.dtypes      # mêmes colonnes, dans le même ordre, avec les mêmes types
    print(f"{nom:9} {relu.count():>7} lignes   {len(relu.columns)} colonnes   mêmes types : {memes_types}")

sales      667579 lignes   25 colonnes   mêmes types : True
products    17820 lignes   24 colonnes   mêmes types : True


stores         12 lignes   30 colonnes   mêmes types : True
dates      109573 lignes   29 colonnes   mêmes types : True


In [10]:
spark.read.parquet(str(SILVER / "sales")).select(
    "ss_ticket_number", "ss_item_sk", "ss_quantity", "ss_sales_price", "ss_net_paid", "date_traitement"
).show(5)

+----------------+----------+-----------+--------------+-----------+--------------------+
|ss_ticket_number|ss_item_sk|ss_quantity|ss_sales_price|ss_net_paid|     date_traitement|
+----------------+----------+-----------+--------------+-----------+--------------------+
|               0|     10000|         54|         83.34|    4500.36|2026-11-18 13:58:...|
|               1|      4597|         20|        133.40|    2668.00|2026-11-18 13:58:...|
|               1|      4517|          6|          4.24|      25.44|2026-11-18 13:58:...|
|               1|      2555|         57|         52.37|    2985.09|2026-11-18 13:58:...|
|               4|     12525|          3|         52.15|     156.45|2026-11-18 13:58:...|
+----------------+----------+-----------+--------------+-----------+--------------------+
only showing top 5 rows


Les 4 jeux Silver se relisent bien, avec le même nombre de lignes et les mêmes types qu'avant l'écriture (Parquet garde les types, contrairement au fichier texte).
Je compare les types (`dtypes`) et pas le schéma complet, parce qu'à la relecture Parquet marque toutes les colonnes comme pouvant être nulles, même `source_fichier` et `date_traitement`.
Ce sont ces données que j'utilise pour le Gold dans le notebook 03.